| Feature            | SwinUNETR                        | SegFormer                                 |
|-------------------|---------------------------------|------------------------------------------|
| Encoder type       | Swin Transformer                 | MiT (Mix Transformer, lightweight)       |
| Decoder type       | CNN-style (UNet-style)           | Transformer-style, lightweight           |
| Architecture type  | Encoder-decoder (UNet-style)     | Fully transformer-based                  |
| Efficiency         | Moderate                         | Very high (lightweight)                  |
| Use case           | Medical image segmentation       | General semantic segmentation            |
| Fully Transformer? | No (only encoder)                | Yes (encoder + decoder)                  |
| Key point          | UNet structure with Swin encoder | Optimized lightweight transformer for segmentation |

In [1]:
from monai.networks import nets
from monai import losses
print(dir(losses))

d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['AsymmetricUnifiedFocalLoss', 'BarlowTwinsLoss', 'BendingEnergyLoss', 'BoxGIoULoss', 'ContrastiveLoss', 'DeepSupervisionLoss', 'Dice', 'DiceCELoss', 'DiceFocalLoss', 'DiceLoss', 'DiffusionLoss', 'FocalLoss', 'GeneralizedDiceFocalLoss', 'GeneralizedDiceLoss', 'GeneralizedWassersteinDiceLoss', 'GlobalMutualInformationLoss', 'HausdorffDTLoss', 'JukeboxLoss', 'LocalNormalizedCrossCorrelationLoss', 'LogHausdorffDTLoss', 'MaskedDiceLoss', 'MaskedLoss', 'MultiScaleLoss', 'NACLLoss', 'PatchAdversarialLoss', 'PerceptualLoss', 'SSIMLoss', 'SURELoss', 'SoftDiceclDiceLoss', 'SoftclDiceLoss', 'TverskyLoss', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', 'adversarial_loss', 'annotations', 'barlow_twins', 'cldice', 'contrastive', 'deform', 'dice', 'dice_ce', 'dice_focal', 'ds_loss', 'focal_loss', 'generalized_dice', 'generalized_dice_focal', 'generalized_wasserstein_dice', 'giou', 'giou_loss', 'hausdorff_loss', 'image_dissimilari

In [1]:
import sys
import os

sys.path.append(os.path.abspath(".."))
from src.dataset import ICRDataset, HFMonaiWrapper
from src.model import load_segformer

In [2]:
model_name = "nvidia/segformer-b1-finetuned-ade-512-512"
model, processor = load_segformer(model_name, num_labels = 1)

dataset_builder = ICRDataset(
    image_dir="../data/test_images",
    mask_dir="../data/test_labels",
    channel=0,
    img_size=512,
    apply_repeat=True,
    apply_clahe=True,
    batch_size=16
)

dataset = dataset_builder.get_dataset()
train_loader = dataset_builder.get_dataloader(transforms_random=None)
hf_dataset = HFMonaiWrapper(dataset, processor, transforms = None)

Loading weights: 100%|██████████| 208/208 [00:01<00:00, 112.48it/s, Materializing param=segformer.encoder.patch_embeddings.3.proj.weight]            
SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b1-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([1, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([1])                      

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR

## Entrainement du model Segformer

Data set kagle

image_train_dir: "/kaggle/input/datasets/borelgoudjou/icr-retina/train_images"
mask_train_dir: "/kaggle/input/datasets/borelgoudjou/icr-retina/train_labels"
image_val_dir: "/kaggle/input/datasets/borelgoudjou/icr-retina/test_images"
mask_val_dir: "/kaggle/input/datasets/borelgoudjou/icr-retina/test_labels"

In [2]:
from transformers import TrainingArguments, TrainerCallback
from src.train import train_segformer
from monai.losses import DiceFocalLoss
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingLR
import yaml

from monai.transforms import (
    Compose, RandFlipd, RandRotate90d, RandGaussianNoised, ToTensord
)

In [3]:
class LrLoggerCallback(TrainerCallback):
    def on_epoch_begin(self, args, state, control, **kwargs):
        optimizer = kwargs['optimizer']
        lrs = [group['lr'] for group in optimizer.param_groups]
        print(f"Epoch {int(state.epoch)} - Learning rates: {lrs}")

In [4]:
def compute_loss(model, inputs, return_outputs=False, **kwargs):
    loss_fc = DiceFocalLoss(sigmoid=True)
    labels = inputs.pop("labels").float()
    outputs = model(**inputs)
    logits = outputs.logits

    # resize labels to match logits --> out of (128*128)
    if labels.shape[2:] != logits.shape[2:]:
        labels = F.interpolate(labels, size=logits.shape[2:], mode="nearest")

    loss = loss_fc(logits, labels)
    return (loss, outputs) if return_outputs else loss

In [5]:
# Charger la configuration YAML
with open("config.yaml", "r") as f:
    config_dataset = yaml.safe_load(f)

transforms_random = Compose([
    RandFlipd(keys=["image", "label"], prob=0.5, spatial_axis=1),
    RandRotate90d(keys=["image", "label"], prob=0.5),
    RandGaussianNoised(keys=["image"], prob=0.2, std=0.01),
    ToTensord(keys=["image", "label"]),
])

training_args = TrainingArguments(
    output_dir="./results",
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=5,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=100,
    learning_rate=1e-4,
    remove_unused_columns=False, # IMPORTANT for segmentation
    fp16=False,   # ← IMPORTANT for CPU
    use_cpu=True,

    # Save best model
    load_best_model_at_end=True,  
    save_total_limit=1,
    metric_for_best_model="mIoU"           
)

trainer, model, hf_dataset_train = train_segformer(
    config_dataset=config_dataset,
    training_args=training_args,
    transforms=None,
    callbacks = [LrLoggerCallback()]
)

trainer.compute_loss = compute_loss

# Nombre total de steps
"""
total_steps = 1 * training_args.num_train_epochs
trainer.lr_scheduler = CosineAnnealingLR(
    optimizer=trainer.optimizer, 
    T_max=total_steps, 
    eta_min=1e-6  # <- LR minimum > 0
)
"""

Loading weights:   0%|          | 0/208 [00:00<?, ?it/s]

SegformerForSemanticSegmentation LOAD REPORT from: nvidia/segformer-b1-finetuned-ade-512-512
Key                           | Status   |                                                                                                     
------------------------------+----------+-----------------------------------------------------------------------------------------------------
decode_head.classifier.weight | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150, 256, 1, 1]) vs model:torch.Size([1, 256, 1, 1])
decode_head.classifier.bias   | MISMATCH | Reinit due to size mismatch - ckpt: torch.Size([150]) vs model:torch.Size([1])                      

Notes:
- MISMATCH	:ckpt weights were loaded, but they did not match the original empty weight shapes.
The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type', 'reduce_labels'


Total params: 13,677,505
Trainable params: 12,032,193
Fraction trainable: 87.97%


Loading dataset: 100%|██████████| 112/112 [00:23<00:00,  4.78it/s]


'\ntotal_steps = 1 * training_args.num_train_epochs\ntrainer.lr_scheduler = CosineAnnealingLR(\n    optimizer=trainer.optimizer, \n    T_max=total_steps, \n    eta_min=1e-6  # <- LR minimum > 0\n)\n'

In [ ]:
trainer.train()

Epoch 0 - Learning rates: [0.0001, 0.0001]


## Train SwinUNETR model

In [ ]:
from monai.networks.nets import SwinUNETR
# Configurer loss, optimizer et métriques
from monai.losses import DiceFocalLoss, DiceLoss
from monai.metrics import DiceMetric
import torch.optim as optim
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = SwinUNETR(
    in_channels=1,
    out_channels=1,
    feature_size=48,
    spatial_dims=2,
    use_checkpoint=True
).to(device)

# Optimizer
optimizer = optim.AdamW(model.parameters(), lr=1e-4)
# Loss
#loss_function = DiceFocalLoss(to_onehot_y=True, softmax=True)
loss_function = DiceLoss(
    smooth_nr=1e-5, 
    smooth_dr=1e-5, 
    squared_pred=True, 
    to_onehot_y=True, 
    softmax=True  # <--- TRÈS IMPORTANT si votre modèle sort 2 canaux
)

In [46]:
#MONAI Trainer attend une fonction prepare_batch pour déplacer les données sur le device et décomposer dictionnaire:
train_builder = ICRDataset(
    image_dir="../data/train_images",
    mask_dir="../data/train_labels",
    channel=1,
    img_size=224,
    cache_rate = 0.05,
    apply_repeat=False,
    apply_clahe=True,
    batch_size=2
)

val_builder = ICRDataset(
    image_dir="../data/test_images",
    mask_dir="../data/test_labels",
    channel=1,
    img_size=224,
    cache_rate = 0.05,
    apply_repeat=False,
    apply_clahe=True,
    batch_size=2
)

dataset_train = train_builder.get_dataset()
dataset_val = val_builder.get_dataset()

train_loader = train_builder.get_dataloader(transforms_random=None)
val_loader = val_builder.get_dataloader(shuffle = False, transforms_random=None)

Loading dataset:   0%|          | 0/13 [00:00<?, ?it/s]

Loading dataset: 100%|██████████| 5/5 [00:00<00:00, 30.74it/s]


In [53]:
len(dataset_train)

268

In [50]:
import torch
from ignite.metrics import Metric
from monai.engines import SupervisedEvaluator
from monai.metrics import MeanIoU, DiceMetric
from monai.transforms import Compose, Activations, AsDiscrete

# 1. Le "Wrapper" Magique (Fait le pont entre PyTorch Ignite et MONAI)
class MonaiIgniteMetric(Metric):
    def __init__(self, monai_metric):
        self.monai_metric = monai_metric
        super().__init__()

    def reset(self):
        # Remet le compteur à zéro au début de l'époque
        self.monai_metric.reset()

    def update(self, output):
        # D'après votre code manuel, l'output est une liste de dictionnaires
        # On extrait proprement les prédictions et les labels
        if isinstance(output, list):
            preds = [x["pred"] for x in output]
            labels = [x["label"] for x in output]
        else:
            preds, labels = output["pred"], output["label"]
            
        # On donne les données à la métrique MONAI pour qu'elle accumule le score
        self.monai_metric(y_pred=preds, y=labels)

    def compute(self):
        # Calcule la moyenne finale à la fin de l'évaluation
        res = self.monai_metric.aggregate()
        return res.item() if hasattr(res, 'item') else res

# 2. Post-traitements
post_pred = Compose([Activations(softmax=True), AsDiscrete(argmax=True, to_onehot=2)])
# (Optionnel : assurez-vous que post_label a la même structure si besoin)

# 3. On déclare les métriques pures et on les "emballe" dans notre pont !
val_metrics = {
    "mIoU": MonaiIgniteMetric(MeanIoU(include_background=True, reduction="mean")),
    "Dice": MonaiIgniteMetric(DiceMetric(include_background=False, reduction="mean"))
}

"""
def prepare_batch(batch, device, non_blocking=False):
    inputs = batch["image"].to(device)
    labels = batch["label"].to(device)
    return inputs, labels
"""

'\ndef prepare_batch(batch, device, non_blocking=False):\n    inputs = batch["image"].to(device)\n    labels = batch["label"].to(device)\n    return inputs, labels\n'

In [51]:
from monai.engines import SupervisedTrainer, SupervisedEvaluator
from monai.engines.utils import default_prepare_batch
from monai.handlers import StatsHandler, CheckpointSaver, ValidationHandler

# 1. Le Trainer
trainer = SupervisedTrainer(
    device=device,
    max_epochs=1,
    train_data_loader=train_loader,
    network=model,
    optimizer=optimizer,
    loss_function=loss_function,
    prepare_batch=default_prepare_batch 
)

# 2. L'Evaluator
evaluator = SupervisedEvaluator(
    device=device,
    val_data_loader=val_loader,
    network=model,
    prepare_batch=default_prepare_batch,
    postprocessing=post_pred,
    key_val_metric={"mIoU": val_metrics["mIoU"]}, 
    additional_metrics={"Dice": val_metrics["Dice"]}
)

# --- ATTACHEMENT DES HANDLERS ---

# 3. Lancer l'évaluation à la fin de chaque époque d'entraînement
ValidationHandler(validator=evaluator, interval=1, epoch_level=True).attach(trainer)

# 4. Affichage propre des logs (Loss pour le trainer, mIoU/Dice pour l'evaluator)
StatsHandler(name="Trainer", output_transform=lambda x: x[0]["loss"] if isinstance(x, list) else x["loss"]).attach(trainer)
StatsHandler(name="Evaluator").attach(evaluator)

# 5. Sauvegarde du MEILLEUR modèle (Attaché à l'evaluator !)
checkpoint_saver = CheckpointSaver(
    save_dir="./checkpoints",
    save_dict={"model": model},
    save_key_metric=True,       # Active la sauvegarde basée sur la métrique clé
    key_metric_name="mIoU",     # Doit correspondre à la clé dans key_val_metric
    key_metric_n_saved=1        # Garde uniquement le meilleur modèle
)
checkpoint_saver.attach(evaluator)

In [52]:
print("🚀 Début de l'entraînement...")
trainer.run()
print("✅ Entraînement terminé.")

🚀 Début de l'entraînement...


single channel prediction, `to_onehot_y=True` ignored.
single channel prediction, `softmax=True` ignored.


2026-03-04 13:47:36,333 - INFO - Epoch: 1/1, Iter: 1/134 -- Loss: -4.7938 
2026-03-04 13:47:46,166 - INFO - Epoch: 1/1, Iter: 2/134 -- Loss: -4.7106 
2026-03-04 13:47:57,203 - INFO - Epoch: 1/1, Iter: 3/134 -- Loss: -5.9726 
2026-03-04 13:48:08,798 - INFO - Epoch: 1/1, Iter: 4/134 -- Loss: -6.0017 
2026-03-04 13:48:18,281 - INFO - Epoch: 1/1, Iter: 5/134 -- Loss: -7.8116 
2026-03-04 13:48:26,783 - INFO - Epoch: 1/1, Iter: 6/134 -- Loss: -5.0447 
2026-03-04 13:48:37,202 - INFO - Epoch: 1/1, Iter: 7/134 -- Loss: -5.3300 


Engine run is terminating due to exception: 


2026-03-04 13:48:39,467 - ERROR - Exception: 
Traceback (most recent call last):
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\ignite\engine\engine.py", line 959, in _internal_run_as_gen
    epoch_time_taken += yield from self._run_once_on_dataset_as_gen()
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\ignite\engine\engine.py", line 1068, in _run_once_on_dataset_as_gen
    self.state.output = self._process_function(self, self.state.batch)
                        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\monai\engines\trainer.py", line 259, in _iteration
    _compute_pred_loss()
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_

KeyboardInterrupt: 

In [ ]:
# 5. Exécution
evaluator.run()

# 6. Affichage final
print(f"Validation mIoU: {evaluator.state.metrics['mIoU']:.4f}")
print(f"Validation Dice: {evaluator.state.metrics['Dice']:.4f}")

Current run is terminating due to exception: applying transform <monai.transforms.compose.Compose object at 0x0000025C13D09DC0>


2026-03-04 13:31:26,299 - ERROR - Exception: applying transform <monai.transforms.compose.Compose object at 0x0000025C13D09DC0>
Traceback (most recent call last):
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\monai\transforms\transform.py", line 150, in apply_transform
    return _apply_transform(transform, data, unpack_items, lazy, overrides, log_stats)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\monai\transforms\transform.py", line 98, in _apply_transform
    return transform(data, lazy=lazy) if isinstance(transform, LazyTrait) else transform(data)
                                                                               ^^^^^^^^^^^^^^^
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\monai\tra

Engine run is terminating due to exception: applying transform <monai.transforms.compose.Compose object at 0x0000025C13D09DC0>


2026-03-04 13:31:26,306 - ERROR - Exception: applying transform <monai.transforms.compose.Compose object at 0x0000025C13D09DC0>
Traceback (most recent call last):
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\monai\transforms\transform.py", line 150, in apply_transform
    return _apply_transform(transform, data, unpack_items, lazy, overrides, log_stats)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\monai\transforms\transform.py", line 98, in _apply_transform
    return transform(data, lazy=lazy) if isinstance(transform, LazyTrait) else transform(data)
                                                                               ^^^^^^^^^^^^^^^
  File "d:\marchine_learning\Projet\Deep_learning Project\Segmentation\segmentatio_ICR\icr_venv\Lib\site-packages\monai\tra

RuntimeError: applying transform <monai.transforms.compose.Compose object at 0x0000025C13D09DC0>

Identify segmentation models

Segmentation models typically have keywords like Seg, UNet, or SPADE in their names. From your list:

Segmentation models:

SegResNet

SegResNetDS

SegResNetDS2

SegResNetVAE

SPADEAutoencoderKL

SPADEDiffusionModelUNet

SPADENet

SwinUNETR

All the SE* / SERes* / SENet* models are classification/backbone models, not segmentation models.